# Chapter 3 Exercises - Run on Google Colab

This notebook is designed to run entirely on Colab's servers.

**Instructions:**
1. Open this file in VS Code
2. Click "Open in Colab" button (top right) or upload to Colab manually
3. Run cells - computation happens on Google's servers, not your PC

In [ ]:
#@title 1. Setup - Clone repo and install dependencies
# This runs on Colab's servers (12GB+ RAM, optional GPU)

!git clone https://github.com/YOUR_USERNAME/scaling-octo-broccoli.git 2>/dev/null || echo "Repo exists"
%cd /content/scaling-octo-broccoli
%pip install -q -e . 

# Add src to path
import sys
sys.path.insert(0, '/content/scaling-octo-broccoli')

print("Setup complete! Colab RAM:", end=" ")
!free -h | grep Mem | awk '{print $2}'

In [ ]:
#@title 2. Upload data files (run once)
# Upload mnist_784.csv.zip from your local machine

from google.colab import files
import os

os.makedirs('data/raw', exist_ok=True)

if not os.path.exists('data/raw/mnist_784.csv.zip'):
    print("Upload mnist_784.csv.zip:")
    uploaded = files.upload()
    for name in uploaded:
        os.rename(name, f'data/raw/{name}')
else:
    print("MNIST data already uploaded")

!ls -lh data/raw/

---
## Exercise 1: MNIST KNN Classifier (>97% accuracy)

Grid search for best hyperparameters. This is the RAM-intensive part that crashed your PC.

In [ ]:
import pandas as pd
import numpy as np
import zipfile
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load MNIST
with zipfile.ZipFile('data/raw/mnist_784.csv.zip', 'r') as z:
    z.extractall('data/raw/')

mnist = pd.read_csv('data/raw/mnist_784.csv')
print(f"Dataset shape: {mnist.shape}")
print(f"Memory usage: {mnist.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# Stratified split
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in splitter.split(mnist.drop('class', axis=1), mnist['class']):
    X_train = mnist.loc[train_idx].drop('class', axis=1)
    y_train = mnist.loc[train_idx]['class']
    X_test = mnist.loc[test_idx].drop('class', axis=1)
    y_test = mnist.loc[test_idx]['class']

print(f"Training: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
%%time
# Grid Search - runs on Colab's 12GB RAM
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 10],
    'weights': ['uniform', 'distance']
}

knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid, cv=4, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate on test set
y_pred = grid_search.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test set accuracy: {accuracy:.4f}")

---
## Exercise 2: Data Augmentation with Image Shifts

In [ ]:
from scipy.ndimage import shift

def shift_image(image, dx, dy):
    """Shift image by dx, dy pixels."""
    return shift(image.reshape(28, 28), [dy, dx], cval=0, mode='constant').flatten()

# Create augmented dataset (5x original size)
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()

X_augmented = list(X_train_np)
y_augmented = list(y_train_np)

shifts = [(1, 0), (-1, 0), (0, 1), (0, -1)]

for dx, dy in shifts:
    print(f"Applying shift ({dx}, {dy})...")
    for img, label in zip(X_train_np, y_train_np):
        X_augmented.append(shift_image(img, dx, dy))
        y_augmented.append(label)

X_augmented = np.array(X_augmented)
y_augmented = np.array(y_augmented)

# Shuffle
rng = np.random.default_rng(42)
shuffle_idx = rng.permutation(len(X_augmented))
X_augmented = X_augmented[shuffle_idx]
y_augmented = y_augmented[shuffle_idx]

print(f"\nOriginal: {len(X_train_np)}, Augmented: {len(X_augmented)}")

In [ ]:
%%time
# Train with best params on augmented data
best_params = grid_search.best_params_
knn_augmented = KNeighborsClassifier(**best_params)
knn_augmented.fit(X_augmented, y_augmented)

# Evaluate
y_pred_aug = knn_augmented.predict(X_test)
accuracy_aug = accuracy_score(y_test, y_pred_aug)
print(f"Test accuracy with augmentation: {accuracy_aug:.4f}")

---
## Exercise 3: Titanic Classification

In [ ]:
import tarfile
from urllib.request import urlretrieve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# Download Titanic data
urlretrieve('https://homl.info/titanic.tgz', 'data/raw/titanic.tgz')
with tarfile.open('data/raw/titanic.tgz') as tar:
    tar.extractall('data/raw/', filter='data')

train_data = pd.read_csv('data/raw/titanic/train.csv', index_col='PassengerId')
print(f"Titanic training samples: {len(train_data)}")
train_data.head()

In [ ]:
# Preprocessing pipeline
X_titanic = train_data.drop('Survived', axis=1)
y_titanic = train_data['Survived']

num_features = ['Age', 'SibSp', 'Parch', 'Fare']
cat_features = ['Pclass', 'Sex', 'Embarked']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

X_processed = preprocessor.fit_transform(X_titanic)
print(f"Processed shape: {X_processed.shape}")

In [ ]:
# Compare classifiers
classifiers = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(gamma='auto'),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

print("10-Fold Cross-Validation Results:")
print("-" * 45)

for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_processed, y_titanic, cv=10, scoring='accuracy')
    print(f"{name:15} | Mean: {scores.mean():.4f} | Std: {scores.std():.4f}")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt
import seaborn as sns

# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = train_data.select_dtypes(include=['number']).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Titanic Feature Correlations')
plt.tight_layout()
plt.show()

---
## Summary

| Exercise | Result |
|----------|--------|
| Ex 1: MNIST KNN | Best: n_neighbors=3, weights=distance → ~97.3% accuracy |
| Ex 2: Augmentation | 5x training data with pixel shifts |
| Ex 3: Titanic | SVM ~82.5% > RF ~81.4% > KNN ~80.7% |